# House Image Discovery with Cost Tracking (Reference Implementation)

This notebook shows that we can discover, frame, and triangulate a target house across panoramic
rosettes keyed on `capture_id` using BigQuery geospatial framing (`rosette_sql`), calibrated
Kannala-Brandt perspective rendering, structured Gemini perception (`HouseView`), and one validated
agentic-vision (`code_execution=True`) storey-count measurement cross-checked locally.

> **Runtime & Cost Banner:** Expected runtime **<= 6 min** on Colab CPU · Gemini ceiling
> **<= $0.15** (`<= 12` plain calls + `1` code-exec call) · Cold BigQuery scan **1.83 GB** at
> **$6.25/TB** (~$0.011 first run, **0 B** on cached re-runs within 35 days).

## 1. Problem, Data Model & Division of Labour

**Problem.** Given approximate `(LAT, LNG)` coordinates of a building, select non-occluded views
from nearby drive sequences, verify visibility and facade attributes (`stories`,
`exterior_material`, `roof_type`), and triangulate the facade location from 2+ camera rays.
Results measure multi-view consistency, not ground-truth accuracy.

**Data model & BigQuery cost.** Every rosette of 7 frames (`_0`..`_6`) is keyed by `capture_id`;
`pano_id` is nullable attribution metadata (~69% of rosettes have `pano_id IS NULL`). Cold scan of
`rosette_sql` is **1.83 GB** at **$6.25/TB** (cached locally up to 35 days).

**Division of labour.** BigQuery computes drive sequences, bearings, and target distances; Python
handles fisheye projection, framing, facade-edge triangulation, and attribute fusion; Gemini plain
predicts `HouseView`; one agentic vision cell counts facade window rows in the Python sandbox.

**Terms of use & authentication.** Imagery comes from your Street View Insights dataset and is
governed by the terms of your Google Cloud agreement: do not redistribute frames or rendered crops
outside your organisation, keep the `"Imagery © Google"` attribution on every figure and map, and
do not persist raw imagery or `gcs_uri` values beyond short-lived working caches (<= 24 h for
frames, <= 35 days for snapshot-keyed metadata).

- **Colab:** calls `google.colab.auth.authenticate_user()`.
- **Local Jupyter:** run `gcloud auth application-default login` once.
- **Workstation helpers:** set `SVI_USE_GCLOUD_TOKEN=1` and `SVI_ECP_PROXY_URL` when required.
- **IAM roles needed:** `roles/bigquery.jobUser`, `roles/bigquery.dataViewer`,
  `roles/storage.objectViewer`, and `roles/aiplatform.user`.

In [ ]:
# Install the shared `svi_geo` helper package (geometry, sequencing, dedup, Gemini client).
# Dependency ranges in pyproject.toml are compatible with Colab preinstalled versions.
# In a clone of the repo the local copy is used: $SVI_GEO_LOCAL_PATH, or the first parent
# directory that contains street_view_insights/panoramic/svi_geo. Otherwise pip installs it
# from GitHub at SVI_GEO_REF.
import importlib.util
import os
import pathlib
import subprocess
import sys

# TODO(after-merge): set to "main" once svi_geo is merged. Override with $SVI_GEO_REF.
SVI_GEO_REF = os.environ.get("SVI_GEO_REF", "94b9530")


def find_local_svi_geo():
    if os.environ.get("SVI_GEO_LOCAL_PATH"):
        return os.environ["SVI_GEO_LOCAL_PATH"]
    here = pathlib.Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "street_view_insights/panoramic/svi_geo" / "pyproject.toml").is_file():
            return str(d / "street_view_insights/panoramic/svi_geo")
    return None


LOCAL_SVI_GEO = find_local_svi_geo()
if importlib.util.find_spec("svi_geo") is None:
    if LOCAL_SVI_GEO:
        src = f"{LOCAL_SVI_GEO}[notebooks]"
    else:
        src = (
            "svi-geo[notebooks] @ git+https://github.com/googlemaps-samples/insights-samples"
            f"@{SVI_GEO_REF}#subdirectory=street_view_insights/panoramic/svi_geo"
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", src])

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
from google.cloud import storage

from svi_geo import attribution, auth, config, data, images, maps, rosette, schemas, usecases, views
from svi_geo import gemini_client as gc

try:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()
except ImportError:
    pass

PROJECT_ID = os.environ.get("PROJECT_ID", "YOUR_PROJECT_ID")
DATASET_ID = "imagery_insights___us"
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")
SNAPSHOT_ID = None
INCLUDE_UNPUBLISHED_PANOS = True
MODEL = gc.DEFAULT_MODEL
LOCATION = "global"
THINKING_LEVEL = "LOW"
MEDIA_RESOLUTION = "MEDIUM"
ENABLE_CODE_EXECUTION = True
MAX_GEMINI_CALLS = 25
MAX_USD = 0.15
CONCURRENCY = 8
SEED = 0
LAT, LNG = 28.05047, -81.96015
RADIUS_M = 80.0
MAX_VIEWS = 8
MAX_PER_SEQ = 4
VIEW_W, VIEW_H = 1024, 768
OUT_DIR = pathlib.Path(os.environ.get("OUT_DIR", "~/tmp/svi_qa/uc1_house")).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

settings = config.resolve_settings(PROJECT_ID, GCS_BUCKET, env=dict(os.environ), dataset=DATASET_ID)
PROJECT_ID, BUCKET = settings.project, settings.bucket
creds = auth.get_credentials()
TABLE = data.pano_table(PROJECT_ID, DATASET_ID)
SNAP_TABLE = f"{PROJECT_ID}.{DATASET_ID}.snapshots"
bq = data.QueryRunner(
    data.make_bigquery_client(PROJECT_ID, creds),
    allowed_tables=data.pano_tables(PROJECT_ID, DATASET_ID),
    cache_dir=data.DEFAULT_QUERY_CACHE,
)
fetcher = images.GcsImageFetcher(storage.Client(project=PROJECT_ID, credentials=creds))
runner = gc.GeminiRunner(
    gc.VertexGeminiBackend(
        gc.make_vertex_client(PROJECT_ID, LOCATION, creds),
        model=MODEL,
        thinking_level=THINKING_LEVEL,
        media_resolution=MEDIA_RESOLUTION,
    ),
    max_calls=MAX_GEMINI_CALLS,
    max_usd=MAX_USD,
    concurrency=CONCURRENCY,
)
intr = rosette.load_intrinsics()

In [ ]:
# concept:sql
# Discover live SV_PANO snapshots and run canonical rosette_sql (capture_id key, LAG sequences).
snap_df = bq.run(data.snapshot_catalog_sql(SNAP_TABLE), [])
ROSETTE_SQL = data.rosette_sql(TABLE)
print(ROSETTE_SQL[:360] + "\n...")
params = data.rosette_params(
    lat=LAT,
    lng=LNG,
    radius_m=RADIUS_M,
    snapshot_id=SNAPSHOT_ID,
    include_unpublished=INCLUDE_UNPUBLISHED_PANOS,
)
rosettes = data.attach_target_framing(
    bq.run(ROSETTE_SQL, params, template_name="rosette_sql"), tlat=LAT, tlng=LNG
)
snap_info = data.summarize_snapshots(rosettes, snap_df)
print(snap_info["summary_line"])
frames = data.frames_from_rosettes(rosettes, bucket=BUCKET)
cost_bq = bq.last_cost()
snap_ids = ",".join(s[:8] for s in snap_info["snapshot_ids"]) or "none"
null_cnt = int(rosettes["pano_id"].isna().sum())
print(
    f"snapshots={snap_ids} rosettes={len(rosettes)} null_pano_id={null_cnt} "
    f"sequences={rosettes['seq_id'].nunique()} bq_bytes={cost_bq['bytes']} "
    f"bq_cached={int(cost_bq['cached'])}"
)

In [ ]:
# concept:geometry
# Worked geometry example: select candidate views, check sensor coverage, and rank across drives.
VARIANT = usecases.DEFAULT_VARIANT
print(VARIANT.describe())
cands = views.house_view_candidates(frames, LAT, LNG, intr, aspect=VIEW_W / VIEW_H)
ranked = views.rank_house_views(cands, max_per_seq=MAX_PER_SEQ, n=MAX_VIEWS)
if ranked.empty:
    raise RuntimeError("No candidate house views found within RADIUS_M.")
r0 = ranked.iloc[0].to_dict()
v0 = views.view_for(r0, VIEW_W, VIEW_H)
b0 = rosette.view_black_fraction(intr, r0["camera_pose"], v0, int(r0["cam_k"]))
az0, el0 = v0.pixel_to_bearing(VIEW_W / 2, VIEW_H / 2)
print(
    f"worked_view capture_id={r0['capture_id']} cam_k={r0['cam_k']} "
    f"hfov={v0.hfov_deg:.1f} black={b0:.4f} ray=({az0:.1f},{el0:.1f})"
)

In [ ]:
# concept:prompt_schema
# Prompt & Pydantic schema inline, single-view demo, then scored run via usecases.uc1_run.
print("Prompt:", usecases.UC1_PROMPT_V0)
print("Schema keys:", list(schemas.HouseView.model_json_schema()["properties"]))
out = await usecases.uc1_run(
    ranked, fetcher.fetch, runner, intr, width=VIEW_W, height=VIEW_H, variant=VARIANT
)
runner.check(context="UC1 HouseView")
res = out["res"]
assert len(res) > 0 and int(res["centred"].sum()) >= 1

In [ ]:
# concept:code_exec
# Agentic vision (1 call): count horizontal window rows in sandbox and cross-check against Sobel-y.
centred_idx = [i for i, c in enumerate(out["res"]["centred"]) if c]
best_i = centred_idx[0] if centred_idx else 0
best_view = out["views_out"][best_i]
fused_stories = out["attrs"]["stories"][0]
ce = await usecases.uc1_count_storeys(
    out["crops"][best_i],
    runner,
    box_2d=best_view.box_2d if best_view else None,
    fused_stories=int(fused_stories) if fused_stories else None,
    seed=SEED,
    code_execution=True,
    validator=lambda r: None,
    expect_stdout=usecases.MEASURE_STDOUT_PATTERN,
)
gc.check_code_exec_trace(ce["trace"], expect_stdout=usecases.MEASURE_STDOUT_PATTERN)
assert ce["agreement_line"].startswith("code_exec_agreement=")
print(ce["agreement_line"])
fig_ce, ax_ce = plt.subplots(figsize=(5, 4))
ax_ce.imshow(ce["overlay"][:, :, ::-1])
ax_ce.set_title(f"Agentic Storey Check: {ce['agreement_line']}")
ax_ce.axis("off")
attribution.save_figure(fig_ce, OUT_DIR / "agentic_storeys.png")
plt.close(fig_ce)

In [ ]:
# concept:validator
# Deterministic validation gates, triangulation residual, and fused house attributes.
loc = out["location"]
print(
    f"black_fraction_max={out['black_fraction_max']:.4f} "
    f"dark_pixel_max={out['dark_pixel_max']:.4f} "
    f"redaction_overlap_max={out['redaction_overlap_max']:.4f}"
)
print(f"centred_views={int(res['centred'].sum())} status={out['status']} attrs={out['attrs']}")
if loc is not None:
    print(f"triangulated=({loc.lat:.6f}, {loc.lng:.6f}) rms_m={loc.rms_m:.2f}")

In [ ]:
# concept:map
# Sighting-ray triangulation map from BigQuery WKT and QA overlay grid with attribution.
m = maps.uc1_house_map(
    rosettes,
    target_lat=LAT,
    target_lng=LNG,
    sightings=out["sightings"],
    location=out["location"],
    out_path=OUT_DIR / "map.html",
)
fig, axes = plt.subplots(1, min(3, len(out["crops"])), figsize=(12, 3.6))
for idx, ax in enumerate(np.atleast_1d(axes)):
    ax.imshow(out["crops"][idx][:, :, ::-1])
    v = out["views_out"][idx]
    status_lbl = "CENTRED" if res.iloc[idx]["centred"] else "REJECTED"
    color = "#2ea043" if status_lbl == "CENTRED" else "#d93025"
    if v and v.box_2d:
        x0, y0, x1, y1 = schemas.box_2d_to_pixels(v.box_2d, VIEW_W, VIEW_H)
        ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], color=color, lw=2)
    ax.set_title(f"{status_lbl} ({res.iloc[idx]['dist_m']:.0f} m)")
    ax.axis("off")
    attribution.add_to_axes(ax)
fig.savefig(OUT_DIR / "overlay_grid.png", dpi=120, bbox_inches="tight")
plt.close(fig)

In [ ]:
# concept:cost
# Token preview across media_resolution levels, BigQuery + Gemini cost summary, and QA metrics.
prev_low = gc.preview_tokens(runner.backend, [usecases.UC1_PROMPT_V0, out["crops"][0]])
est = gc.estimate_cost(len(out["crops"]), preview=prev_low)
print(f"preview_tokens={prev_low} pre_run_est_usd=${est:.4f} ceiling=${MAX_USD:.2f}")
print(runner.cost.summary(), runner.cost.summary(ceiling=MAX_USD))
(OUT_DIR / "metrics.json").write_text(
    json.dumps(
        {
            "notebook": "house_image_discovery_with_cost",
            "rosettes": int(len(rosettes)),
            "null_pano_id": int(rosettes["pano_id"].isna().sum()),
            "centred_views": int(res["centred"].sum()),
            "est_cost_usd": float(runner.cost.usd),
        },
        indent=2,
    ),
    encoding="utf-8",
)
artefacts = len(list(OUT_DIR.glob("*")))
print(f"map_rendered=1 artefacts={artefacts} peak_rss_mb={images.peak_rss_mb():.0f}")

## 9. Limitations & Next Steps

- Heavy vegetation or privacy redaction blobs (`redaction_overlap > 0.30`) down-weight a view in
  attribute fusion.
- See `analyze_sequential_images.ipynb` for drive-sequence multi-object triangulation and
  `EVALUATION.md` for the 26-metric label-free benchmark across 3 AOIs.